# Factoring 21 with native FlagQuantum

A complete local Shor workflow: exact phase readout, actual samples, continued fractions and verified factors. The quantum example keeps the original ten phase bits and five residue bits.


In [ ]:
import math
import random
from fractions import Fraction
from io import BytesIO

from IPython.display import Image, display

from examples.synthesis.algorithms.number_theory_and_cryptography.shor.overview.classical import (
    convergents,
    postprocess_phase,
)
from examples.synthesis.algorithms.number_theory_and_cryptography.shor.overview.workflow import (
    plot_estimate,
    run_estimate,
)


## Classical warm-up

The original larger number is used only for a classical random-base/gcd demonstration. This is not a claim that the quantum program factors 123456789. An explicit local seed replaces the unseeded source draw.


In [ ]:
large_number = 123456789
base = random.Random(1700).randrange(2, large_number)
common_divisor = math.gcd(base, large_number)
print("Base:", base, "Modulus:", large_number, "GCD:", common_divisor)
if common_divisor > 1:
    print("Classical factors:", common_divisor, large_number // common_divisor)


## Exact continued fractions

Preserve the original 17/1024 example and five float examples. Fraction(float) preserves the exact binary value, matching the source Rational(float) behavior. The denominator cutoff is strictly less than n.


In [ ]:
print(convergents(Fraction(17, 1024)))
for value in (0.1953125, 0.39160156, 0.0, 0.59667969, 0.80078125):
    print(value, "-->", postprocess_phase(Fraction(value), 18))


## Original quantum problem

The existing native qfunc allocates registers, sets the residue to one, applies uniform state preparation, modular exponentiation and inverse QFT. The dropped residue remains in the independent full-state reference; only clean scratch bits are required to return to zero. Readout returns named Fraction labels, with no wire numbering or bitstring decoding.


In [ ]:
estimate = run_estimate(seed=1700)
print("Shots:", estimate.shots)
print(estimate.resources)
print("Complete-state error:", estimate.maximum_state_error)
print("Analytic error:", estimate.analytic_state_error)
print("Dirty clean-work probability:", estimate.dirty_workspace_probability)
for phase, count in estimate.observations[:20]:
    print(phase, count)
print("Retained fractions:", estimate.fractions)
print("Recovered order:", estimate.order)
print("Factors:", estimate.factors)
assert estimate.order == 6
assert math.prod(estimate.factors) == 21


## Measured phase distribution

The histogram uses actual 2048-shot frequencies. The upper axis shows source-filtered continued fractions. The local seed is reproducible; hosted samples were unseeded.


In [ ]:
figure = plot_estimate(estimate)
with BytesIO() as buffer:
    figure.savefig(buffer, format="png")
    display(Image(data=buffer.getvalue()))
figure.clear()


## Explicit source corrections

The original helper writes Python XOR (`^`) where mathematical exponentiation is intended. The native helper uses modular `pow`, rejects invalid orders and checks nontrivial factors. The source parameters coincidentally give 3 and 7 under XOR, but valid alternatives such as base 8 modulo 21 expose the error. The native workflow derives the period from the LCM of retained denominators, then checks the original r=6 assertion; it does not silently hard-code the answer.
